# Exercise 9 — Sound and music description

In this exercise you describe sounds using simple machine-learning workflows with Freesound descriptors. You will use the Freesound API to download short instrumental sounds and pre-computed descriptors, then apply unsupervised and supervised methods for organisation and classification.

You will complete **four parts**:
1. Download sound collections and descriptor files from Freesound.
2. Select a descriptor pair for 2D visual clustering.
3. Cluster sounds with k-means using multiple descriptors.
4. Classify query sounds with k-NN and interpret the result.

The notebook already provides most of the implementation, so focus on selecting good sounds, choosing meaningful descriptors, and explaining your decisions.

### Relevant concepts

#### Freesound API
The Freesound API allows programmatic search and retrieval of sounds, metadata, and analysis descriptors. It supports text queries, tag filters, and duration constraints, similar to advanced search on the website. Documentation: https://freesound.org/docs/api/

#### Sound descriptors
You will use descriptors pre-computed with Essentia and stored in Freesound. These descriptors summarise spectral and temporal characteristics (e.g., spectral centroid, dissonance, attack time, inharmonicity, MFCCs), which can be used as features for clustering and classification.

#### Euclidean distance
The Euclidean distance between two descriptor vectors $p$ and $q$ is:

$$
d(p,q) = \sqrt{\sum_{i=1}^{n}(q_i - p_i)^2}
$$

This distance is used as a similarity measure in both clustering and k-NN classification.

#### K-means clustering
K-means partitions data into $k$ clusters by minimising within-cluster variance. Each sound is assigned to the nearest centroid in feature space. For this exercise, use $k=3$ (one cluster per chosen instrument class).

#### K-nearest neighbours (k-NN)
k-NN predicts the class of a query sample by majority vote among its $k$ nearest training sounds in descriptor space. With $k=1$, classification is the label of the nearest neighbour.

## Part 1 – Download sounds and descriptors from Freesound

Download three coherent collections of instrumental sounds (20 sounds per instrument) and their descriptors using the Freesound API.

**Setup requirements:**
- Get a Freesound API key: http://www.freesound.org/apiv2/apply/
- Create an output directory named `testDownload`.
- Install the Freesound Python client (`freesound-python`) and import `freesound`.

You will call `download_sounds_freesound()` with:
1. `queryText`: instrument keyword (e.g., `violin`, `trumpet`, `cello`).
2. `tag`: optional filter (e.g., `single-note`, `multisample`).
3. `duration`: `(min_seconds, max_seconds)` tuple.
4. `API_Key`: your Freesound API key.
5. `outputDir`: target output directory.
6. `topNResults`: number of sounds to download.
7. `featureExt`: descriptor extension (typically `.json`).

**Task (E9-1.1):**
- Select 3 instruments from: violin, guitar, bassoon, trumpet, clarinet, cello, naobo.
- Design search parameters (`queryText`, `tag`, `duration`) so the top 20 results are coherent single notes/strokes and under 10 seconds.
- Download and inspect both the sounds and their descriptor files.

Before running the API script, validate your search query on the Freesound website to ensure top-ranked results are relevant.

In [1]:
import os, sys
import json
import freesound as fs

descriptors = [ 'lowlevel.spectral_centroid.mean',
                'lowlevel.spectral_contrast.mean',
                'lowlevel.dissonance.mean',
                'lowlevel.hfc.mean',
                'lowlevel.mfcc.mean',
                'sfx.logattacktime.mean',
                'sfx.inharmonicity.mean']

In [2]:
def download_sounds_freesound(queryText = "", tag=None, duration=None, API_Key = "", outputDir = "", topNResults = 5, featureExt = '.json'):
  """
  This function downloads sounds and their descriptors from freesound using the queryText and the 
  tag specified in the input. Additionally, you can also specify the duration range to filter sounds 
  based on duration.
  
  Inputs:
        (Input parameters marked with a * are optional)
        queryText (string): query text for the sounds (eg. "violin", "trumpet", "cello", "bassoon" etc.)
        tag* (string): tag to be used for filtering the searched sounds. (eg. "multisample",  
                       "single-note" etc.)
        duration* (tuple): min and the max duration (seconds) of the sound to filter, eg. (0.2,15)
        API_Key (string): your api key, which you can obtain from : www.freesound.org/apiv2/apply/
        outputDir (string): path to the directory where you want to store the sounds and their 
                            descriptors
        topNResults (integer): number of results(sounds) that you want to download 
        featureExt (string): file extension for storing sound descriptors
  output:
        This function downloads sounds and descriptors, and then stores them in outputDir. In 
        outputDir it creates a directory of the same name as that of the queryText. In this 
        directory outputDir/queryText it creates a directory for every sound with the name 
        of the directory as the sound id. Additionally, this function also dumps a text file 
        containing sound-ids and freesound links for all the downloaded sounds in the outputDir. 
        NOTE: If the directory outputDir/queryText exists, it deletes the existing contents 
        and stores only the sounds from the current query. 
  """ 
  
  # Checking for the compulsory input parameters
  if queryText == "":
    print("\n")
    print("Provide a query text to search for sounds")
    return -1
    
  if API_Key == "":
    print("\n")
    print("You need a valid freesound API key to be able to download sounds.")
    print("Please apply for one here: www.freesound.org/apiv2/apply/")
    print("\n")
    return -1
    
  if outputDir == "" or not os.path.exists(outputDir):
    print("\n")
    print("Please provide a valid output directory. This will be the root directory for storing sounds and descriptors")
    return -1    
  
  # Setting up the Freesound client and the authentication key
  fsClnt = fs.FreesoundClient()
  fsClnt.set_token(API_Key,"token")  
  
  # Creating a duration filter string that the Freesound API understands
  if duration and type(duration) == tuple:
    flt_dur = " duration:[" + str(duration[0])+ " TO " +str(duration[1]) + "]"
  else:
    flt_dur = ""
 
  if tag and type(tag) == str:
    flt_tag = "tag:"+tag
  else:
    flt_tag = ""

  # Querying Freesound
  page_size = 30
  if not flt_tag + flt_dur == "":
    qRes = fsClnt.text_search(query=queryText ,filter = flt_tag + flt_dur,sort="score", fields="id,name,previews,username,url,analysis", descriptors=','.join(descriptors), page_size=page_size, normalized=1)
  else:
    qRes = fsClnt.text_search(query=queryText ,sort="score",fields="id,name,previews,username,url,analysis", descriptors=','.join(descriptors), page_size=page_size, normalized=1)
  
  outDir2 = os.path.join(outputDir, queryText)
  if os.path.exists(outDir2):             # If the directory exists, it deletes it and starts fresh
      os.system("rm -r " + outDir2)
  os.mkdir(outDir2)

  pageNo = 1
  sndCnt = 0
  indCnt = 0
  totalSnds = min(qRes.count,200)   # System quits after trying to download after 200 times
  
  # Creating directories to store output and downloading sounds and their descriptors
  downloadedSounds = []
  while(1):
    if indCnt >= totalSnds:
      print("Not able to download required number of sounds. Either there are not enough search results on freesound for your search query and filtering constraints or something is wrong with this script.")
      break
    sound = qRes[indCnt - ((pageNo-1)*page_size)]
    print("Downloading mp3 preview and descriptors for sound with id: %s"%str(sound.id))
    outDir1 = os.path.join(outputDir, queryText, str(sound.id))
    if os.path.exists(outDir1):
      os.system("rm -r " + outDir1)
    os.system("mkdir " + outDir1)
    
    mp3Path = os.path.join(outDir1,  str(sound.previews.preview_lq_mp3.split("/")[-1]))
    ftrPath = mp3Path.replace('.mp3', featureExt)
    
    try:
      
      fs.FSRequest.retrieve(sound.previews.preview_lq_mp3, fsClnt, mp3Path)
      # Initialize a dictionary to store descriptors
      features = {}
      # Obtaining all the descriptors
      for desc in descriptors:
        features[desc]=[]
        features[desc].append(eval("sound.analysis."+desc))
      
      # Once we have all the descriptors, store them in a json file
      json.dump(features, open(ftrPath,'w'))
      sndCnt+=1
      downloadedSounds.append([str(sound.id), sound.url])

    except:
      if os.path.exists(outDir1):
        os.system("rm -r " + outDir1)
    
    indCnt +=1
    
    if indCnt%page_size==0:
      qRes = qRes.next_page()
      pageNo+=1
      
    if sndCnt>=topNResults:
      break
  # Dump the list of files and Freesound links
  fid = open(os.path.join(outDir2, queryText+'_SoundList.txt'), 'w')
  for elem in downloadedSounds:
    fid.write('\t'.join(elem)+'\n')
  fid.close()

In [ ]:
# E9 - 1.1: call download_sounds_freesound for 3 instruments with coherent query settings
### your code here


**E9 – 1.2: Conceptual questions on collection coherence (answer here)**

1. For each of your three instruments, report the final query settings (`queryText`, `tag`, `duration`) and explain why these settings produce coherent single-note/single-stroke samples.
2. Out of the first 20 downloaded sounds per instrument, how many did you consider clearly valid examples of that instrument class? If you rejected any samples, describe the most common reasons (e.g., polyphonic texture, heavy effects, wrong instrument, too long, poor recording quality).
3. Compare at least two query attempts for one instrument (initial vs refined). What changed in the results after modifying `tag` or `duration`, and why did those changes improve coherence?
4. Did you observe descriptor outliers inside any instrument set? Pick one outlier sound ID and explain why its descriptor profile might differ from the rest of its class.
5. Why is collection coherence critical before running Part 2 clustering and Part 3 k-means? Describe one concrete failure mode in clustering/classification that would appear if your downloaded collections include many mislabeled or heterogeneous samples.
6. If you had to improve coherence further without reducing diversity too much, what additional metadata filters would you add (e.g., license, samplerate, channels, tags), and why?

## Part 2 – Select two descriptors for 2D clustering

Choose a pair of descriptors that best separates your three instrument classes in a 2D scatter plot.

Using `targetDir` and `descInput`, plot all sounds from your selected instrument folders with different colours per class. Optionally enable annotation to display Freesound IDs.

- Try multiple descriptor pairs from the mapping.
- Keep only your 3 selected instruments inside `targetDir`.
- Generate the 2D scatter plot and select the pair that gives the clearest separation and most compact within-class grouping.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.cluster.vq import vq, kmeans, whiten

# Mapping of descriptors
descriptorMapping = { 0: 'lowlevel.spectral_centroid.mean',
                      1: 'lowlevel.dissonance.mean',
                      2: 'lowlevel.hfc.mean',
                      3: 'sfx.logattacktime.mean',
                      4: 'sfx.inharmonicity.mean',
                      5: 'lowlevel.spectral_contrast.mean.0',
                      6: 'lowlevel.spectral_contrast.mean.1',
                      7: 'lowlevel.spectral_contrast.mean.2',
                      8: 'lowlevel.spectral_contrast.mean.3',
                      9: 'lowlevel.spectral_contrast.mean.4',
                      10: 'lowlevel.spectral_contrast.mean.5',
                      11: 'lowlevel.mfcc.mean.0',
                      12: 'lowlevel.mfcc.mean.1',
                      13: 'lowlevel.mfcc.mean.2',
                      14: 'lowlevel.mfcc.mean.3',
                      15: 'lowlevel.mfcc.mean.4',
                      16: 'lowlevel.mfcc.mean.5'
                    }

In [ ]:
def convFtrDict2List(ftrDict):
  """
  This function converts descriptor dictionary to an np.array. The order in the numpy array (indices) 
  are same as those mentioned in descriptorMapping dictionary.
  
  Input: 
    ftrDict (dict): dictionary containing descriptors downloaded from the freesound
  Output: 
    ftr (np.ndarray): Numpy array containing the descriptors for processing later on
  """
  ftr = []
  for key in range(len(descriptorMapping.keys())):
    try:
      ftrName, ind = '.'.join(descriptorMapping[key].split('.')[:-1]), int(descriptorMapping[key].split('.')[-1])
      ftr.append(ftrDict[ftrName][0][ind])
    except:
      ftr.append(ftrDict[descriptorMapping[key]][0])
  return np.array(ftr)

def fetchDataDetails(inputDir, descExt = '.json'):
  """
  This function is used by other functions to obtain the information regarding the directory structure 
  and the location of descriptor files for each sound 
  """
  dataDetails = {}
  for path, dname, fnames  in os.walk(inputDir):
    for fname in fnames:
      if descExt in fname.lower():
        remain, rname, cname, sname = path.split('/')[:-3], path.split('/')[-3], path.split('/')[-2], path.split('/')[-1]
        if cname not in dataDetails:
          dataDetails[cname]={}
        fDict = json.load(open(os.path.join('/'.join(remain), rname, cname, sname, fname),'r'))
        dataDetails[cname][sname]={'file': fname, 'feature':fDict}
  return dataDetails

In [ ]:
# E9 - 2.1: select descriptor pair and visualize clustering of the three chosen instruments
inputDir = "testDownload/"

### this is the main line to modify; replace XX with descriptor indices from 0 to 16

descInput = (XX, XX)


# no need to change the code from here
anotOn = 0
dataDetails = fetchDataDetails(inputDir)
colors = ['r', 'g', 'c', 'b', 'k', 'm', 'y']

plt.figure(figsize=(15, 10))

legArray = []
catArray = []
for ii, category in enumerate(dataDetails.keys()):
    catArray.append(category)
    for soundId in dataDetails[category].keys():
        filepath = os.path.join(inputDir, category, soundId, dataDetails[category][soundId]['file'])
        descSound = convFtrDict2List(json.load(open(filepath, 'r')))
        x_cord = descSound[descInput[0]]
        y_cord = descSound[descInput[1]]
        plt.scatter(x_cord, y_cord, c = colors[ii], s=200, alpha=0.75)
        if anotOn==1:
            plt.annotate(soundId, xy=(x_cord, y_cord), xytext=(x_cord, y_cord))
    circ = Line2D([0], [0], linestyle="none", marker="o", alpha=0.75, markersize=10, markerfacecolor=colors[ii])
    legArray.append(circ)
  
plt.ylabel(descriptorMapping[descInput[1]])
plt.xlabel(descriptorMapping[descInput[0]])
plt.legend(legArray,catArray,numpoints=1,bbox_to_anchor=(0.,1.02,1.,.102),loc=3,ncol=len(catArray),mode="expand",borderaxespad=0.)

**E9 – 2.2: Conceptual questions on descriptor choice (answer here)**

1. List at least four descriptor pairs you tested and briefly describe the visual clustering quality of each pair (compactness within class and separation between classes).
2. For the pair you selected, explain what each descriptor measures perceptually (e.g., brightness, attack, noisiness, harmonicity) and why that makes it suitable for separating your three instrument classes.
3. Identify one instrument pair that still overlaps in your selected 2D plot. Why is this overlap expected from acoustics/timbre? Suggest one additional descriptor that could help separate them and justify your choice.
4. Did normalisation or outliers affect your visual interpretation? Pick one outlier point, give its sound ID, and explain whether it is a true timbral exception, a mislabeled sample, or a recording artifact.
5. If you were forced to choose a different descriptor pair (second-best), which one would you pick and why? What trade-off would you accept compared with your best pair?

## Part 3 – Cluster sounds with k-means

Cluster your sounds using more than two descriptors with `cluster_sounds(targetDir, nCluster, descInput)`.

- Use the same dataset from Part 1.
- Set `nCluster = 3` (one cluster per instrument class).
- Start from descriptors that worked in Part 2, then add descriptors only when they improve clustering.
- Test several descriptor subsets and compare resulting accuracy.

Because k-means initialises centroids randomly, results can vary across runs. Report the best result you obtained and explain likely causes of systematic confusions (e.g., consistent overlap between two instruments).

In [ ]:
def cluster_sounds(targetDir, nCluster = -1, descInput=[]):
  """
  This function clusters all the sounds in targetDir using kmeans clustering.
  
  Input:
    targetDir (string): Directory where sound descriptors are stored (all the sounds in this 
                        directory will be used for clustering)
    nCluster (int): Number of clusters to be used for kmeans clustering.
    descInput (list) : List of indices of the descriptors to be used for similarity/distance 
                       computation (see descriptorMapping)
  Output:
    Prints the class of each cluster (computed by a majority vote), number of sounds in each 
    cluster and information (sound-id, sound-class and classification decision) of the sounds 
    in each cluster. Optionally, you can uncomment the return statement to return the same data.
  """
  
  dataDetails = fetchDataDetails(targetDir)
  
  ftrArr = []
  infoArr = []
  
  if nCluster ==-1:
    nCluster = len(dataDetails.keys())
  for cname in dataDetails.keys():
    #iterating over sounds
    for sname in dataDetails[cname].keys():
      ftrArr.append(convFtrDict2List(dataDetails[cname][sname]['feature'])[descInput])
      infoArr.append([sname, cname])
  
  ftrArr = np.array(ftrArr)
  infoArr = np.array(infoArr)
  
  ftrArrWhite = whiten(ftrArr)
  centroids, distortion = kmeans(ftrArrWhite, nCluster)
  clusResults = -1*np.ones(ftrArrWhite.shape[0])
  
  for ii in range(ftrArrWhite.shape[0]):
    diff = centroids - ftrArrWhite[ii,:]
    diff = np.sum(np.power(diff,2), axis = 1)
    indMin = np.argmin(diff)
    clusResults[ii] = indMin
  
  ClusterOut = []
  classCluster = []
  globalDecisions = []  
  for ii in range(nCluster):
    ind = np.where(clusResults==ii)[0]
    freqCnt = []
    for elem in infoArr[ind,1]:
      freqCnt.append(infoArr[ind,1].tolist().count(elem))
    indMax = np.argmax(freqCnt)
    classCluster.append(infoArr[ind,1][indMax])
    
    print("\n(Cluster: " + str(ii) + ") Using majority voting as a criterion this cluster belongs to " + 
          "class: " + classCluster[-1])
    print ("Number of sounds in this cluster are: " + str(len(ind)))
    decisions = []
    for jj in ind:
        if infoArr[jj,1] == classCluster[-1]:
            decisions.append(1)
        else:
            decisions.append(0)
    globalDecisions.extend(decisions)
    print ("sound-id, sound-class, classification decision")
    ClusterOut.append(np.hstack((infoArr[ind],np.array([decisions]).T)))
    print (ClusterOut[-1])
  globalDecisions = np.array(globalDecisions)
  totalSounds = len(globalDecisions)
  nIncorrectClassified = len(np.where(globalDecisions==0)[0])
  print("Out of %d sounds, %d sounds are incorrectly classified considering that one cluster should "
        "ideally contain sounds from only a single class"%(totalSounds, nIncorrectClassified))
  print("You obtain a classification (based on obtained clusters and majority voting) accuracy "
         "of %.2f percentage"%round(float(100.0*float(totalSounds-nIncorrectClassified)/totalSounds),2))
  # return ClusterOut

In [ ]:
# E9 - 3.1: run cluster_sounds with descriptor subsets and compare results
### your code here

**E9 – 3.2: Conceptual questions (answer here)**

1. List the descriptor subsets you tested for k-means (at least three combinations), and report the clustering accuracy for each run. Which subset gave the best result, and what is the minimum number of descriptors that still kept accuracy close to your best run?
2. Inspect the confusion pattern in your best clustering result. Which instrument pair was most frequently mixed up, and which acoustic/timbral properties of those instruments explain that overlap in descriptor space?
3. K-means uses random centroid initialisation. Run your best descriptor configuration at least 5 times and report the mean and range of accuracy. Did performance vary significantly? Based on your results, is your descriptor set robust or sensitive to initialisation?
4. Pick one wrongly clustered sound and listen to it. Does it acoustically resemble the class where it was assigned more than its original class? Use at least two descriptors to justify why its position in feature space may be ambiguous.

## Part 4 – Classify sounds with k-NN

Classify query sounds with `classify_sound_kNN()` using Euclidean distance on selected descriptors.

The query sound should be different from the sounds used to define your three classes (ideally from a different instrument). The goal is to understand *why* a sound is assigned to one class based on nearest-neighbour similarity.

-Download query sounds and descriptors from Freesound.
- Perform at least 5 classification experiments by varying: Query sound, Descriptor subset, `K` (odd positive integers are typical)

Include both expected and surprising/incorrect predictions, then explain each outcome in terms of descriptor-space proximity and timbral similarity.

In [ ]:
def compute_similar_sounds(queryFile, targetDir, descInput = []):
  """
  This function returns similar sounds for a specific queryFile. Given a queryFile this function 
  computes the distance of the query to all the sounds found in the targetDir and sorts them in 
  the increasing order of the distance. This way we can obtain similar sounds to a query sound.
  
  Input:
    queryFile (string): Descriptor file (.json, unless changed)
    targetDir (string): Target directory to search for similar sounds (using their descriptor files)
    descInput (list) : list of indices of the descriptors to be used for similarity/distance computation 
                       (see descriptorMapping)
  Output: 
    List containing an ordered list of similar sounds. 
  """
  
  dataDetails = fetchDataDetails(targetDir)
  
  #reading query feature dictionary
  qFtr = json.load(open(queryFile, 'r'))
  dist = []
  # Iterating over classes
  for cname in dataDetails.keys():
    # Iterating over sounds
    for sname in dataDetails[cname].keys():
      f1 =  convFtrDict2List(qFtr)
      f2 =  convFtrDict2List(dataDetails[cname][sname]['feature'])
      eucDist = np.sqrt(np.sum(np.power(np.array(f1[descInput]) - np.array(f2[descInput]), 2)))
      dist.append([eucDist, sname, cname])
  
  # Sorting the array based on the distance
  indSort = np.argsort(np.array(dist)[:,0])
  return (np.array(dist)[indSort,:]).tolist()

        
def classify_sound_kNN(queryFile, targetDir, K, descInput = []):
  """
  This function performs the KNN classification of a sound. The nearest neighbors are chosen from 
  the sounds in the targetDir.
   
  Input:
    queryFile (string): Descriptor file (.json, unless changed)
    targetDir (string): Target directory to search for similar sounds (using their descriptor files)
    K (int) : Number of nearest neighbors to consider for KNN classification.
    descInput (list) : List of indices of the descriptors to be used for similarity/distance computation 
                      (see descriptorMapping)
  Output:
    predClass (string): Predicted class of the query sound
  """
  distances = compute_similar_sounds(queryFile, targetDir, descInput)
  if len(np.where((np.array(distances)[:,0].astype(np.float64))==0)[0])>0:
    print("Warning: We found an exact copy of the query file in the target directory. "
          "Beware of duplicates while doing KNN classification.")
  
  classes = (np.array(distances)[:K,2]).tolist()
  freqCnt = []
  for ii in range(K):
    freqCnt.append(classes.count(classes[ii]))
  indMax = np.argmax(freqCnt)
  predClass =  classes[indMax]
  print ("This sample belongs to class: " + str(predClass))
  return predClass


In [ ]:
# E9 - 4.1: download one query sound and its descriptors
### your code here

**E9 – 4.2: Classification setup (answer here)**

1. For each of your 5 k-NN experiments, report: query sound ID/link, descriptor subset, value of `K`, and predicted class. Keep this in a compact table.
2. Why did you choose those descriptors for k-NN? For each descriptor, explain what perceptual property it captures (e.g., brightness, noisiness, attack sharpness) and why that property should help discriminate your instrument classes.
3. How did changing `K` affect prediction stability across your experiments? Compare at least one small `K` (e.g., 1 or 3) with one larger `K` (e.g., 7 or 9). Which gave more consistent results, and why?
4. Include at least one query sound that is outside your three training classes. Why is it mapped to the predicted class? Identify which training-class timbral characteristics the query most closely matches.

In [ ]:
# E9 - 4.3: classify the downloaded query sound with classify_sound_kNN
### your code here


**E9 – 4.4: Classification interpretation (answer here)**

1. For one correct prediction and one incorrect/surprising prediction, inspect the nearest neighbours returned by the distance metric. Are the top neighbours perceptually plausible? Explain using both listening and descriptor values.
2. Compute or report the distance gap between the closest and second-closest class neighbourhoods for each case. Was the incorrect prediction a borderline decision (small gap) or a clear mismatch (large gap)?
3. If you could add one more descriptor to improve the weakest experiment, which one would you add (from the mapping) and why? Predict how it would change the decision boundary between the two most confusable classes.
4. Reflect on the limits of descriptor-based k-NN in this setup: what kinds of musical variations (articulation, recording conditions, effects, pitch register) are likely to break classification even if the instrument identity is obvious to a listener?